# Module 7 — Advanced LLM Evaluation Frameworks & Traceability

**Architectural Layer:** LLM Evaluation  
**Toolchain:** DeepEval · RAGAS · Langfuse · Weights & Biases  
**Objective:** Evaluate LLM outputs for faithfulness, relevance, and safety; assess RAG pipeline quality; and trace multi-step agent executions.

---

## 🧠 Why LLM Evaluation Is Fundamentally Different

### The Core Problem

Traditional ML: "Is the prediction correct?" → accuracy = 94.3% ✅  
LLM: "Is this paragraph good?" → ???

There's no single number that captures LLM quality. A response can be:
- ✅ Grammatically perfect but ❌ factually wrong (hallucination)
- ✅ Factually correct but ❌ not answering the question (irrelevant)
- ✅ Relevant answer but ❌ offensive language (toxicity)
- ✅ Safe and relevant but ❌ uses information NOT in the context (unfaithful)

### LLM Evaluation Dimensions

| Dimension | Question | Critical For |
|-----------|---------|-------------|
| **Faithfulness** | Is the response grounded in the provided context? | RAG systems |
| **Answer Relevance** | Does it actually answer the question? | All chatbots |
| **Context Precision** | Did we retrieve the RIGHT documents? | RAG retrieval |
| **Context Recall** | Did we retrieve ALL relevant documents? | RAG retrieval |
| **Toxicity** | Is the response harmful or biased? | Public-facing apps |
| **Hallucination** | Does it contain made-up facts? | Factual systems |

### 🤔 What is \"LLM-as-a-Judge\"?

Since human evaluation is expensive (~$50/hour for expert annotators), we use a STRONGER LLM (e.g., GPT-4o) to evaluate the outputs of a WEAKER LLM (e.g., Phi-2).

```
Phi-2 generates: \"Paris is the capital of Germany\" (wrong)
GPT-4o judges:   \"Faithfulness: 0.0 — Paris is the capital of France, not Germany\"
```

**⚖️ Trade-off:** LLM-as-a-judge is 10x cheaper than humans but ~85% agreement with human evaluators (not perfect).

### ⚖️ Evaluation Tool Comparison

| Tool | Focus | Strengths | Limitations |
|------|-------|-----------|-------------|
| **DeepEval** | Unit tests for LLM outputs | CI/CD integration, assertions | Needs judge LLM |
| **RAGAS** | RAG-specific evaluation | Context metrics, standard | RAG-only |
| **Langfuse** | Observability + tracing | Open-source, prompt versioning | Requires server |
| **W&B Weave** | Tracing + debugging | Great UI, dataset linking | Commercial |
| **Arize Phoenix** | Embeddings + tracing | Local-first, notebook-friendly | Smaller ecosystem |

---

### 🎓 Junior to Senior: Concept Bridge

**The Senior Perspective:** Vibes are not metrics. Eye-balling 10 prompts to see if 'they look better' is amateur science. Seniors establish automated evaluation frameworks (DeepEval, Ragas) to statistically prove a prompt update improved the baseline.

**Mental Model:** LLM-as-a-Judge Eval. Since answers aren't deterministic, you use a superior model (GPT-4) to read the prompt, context, and output, and grade it on a strict 1-10 rubric for Faithfulness and Relevance.

**Common Junior Pitfall:** Deploying a massive prompt change to production because it fixed one specific user bug, only to realize (days later) it severely degraded performance on 50 other historical edge cases ('Regression').

---


In [ ]:
# Cell 1 — Install dependencies
!pip install -q deepeval ragas langfuse pandas numpy datasets

## 1 · DeepEval: Unit Tests for LLM Outputs

### 🤔 What Are \"Unit Tests\" for LLMs?

In traditional software:
```python
assert add(2, 3) == 5  # Deterministic: always true or false
```

For LLMs:
```python
assert faithfulness(response, context) > 0.8  # Probabilistic: scored 0-1
```

DeepEval lets you write these tests and run them in your CI/CD pipeline, blocking deployment if scores are too low.

### 🤔 When to Use DeepEval?

- **Before deployment:** Run test suite against all critical use cases
- **Regression testing:** "Did the new prompt template break anything?"
- **Prompt iteration:** Compare scores between prompt versions
- **Model migration:** "Does the new model perform as well as the old one?"

In [ ]:
# Cell 2 — DeepEval: Define and run LLM test cases
#
# WHAT IS HAPPENING?
# 1. Define test cases with: input, expected output, context, actual output
# 2. Define metrics: Faithfulness (grounded in context?) and AnswerRelevance (answers the question?)
# 3. Score each test case and determine pass/fail
#
# In production, you'd use DeepEval's built-in LLM-as-a-judge.
# Here we simulate the scoring to demonstrate the framework.

import json

# Define test cases: simulate what DeepEval would evaluate
test_cases = [
    {
        "id": "TC-001",
        "input": "What is the company's revenue for Q3 2025?",
        "context": "The company reported Q3 2025 revenue of $52.8M, a 12% increase from Q2.",
        "expected_output": "The company's Q3 2025 revenue was $52.8M.",
        "actual_output": "The company's Q3 2025 revenue was $52.8M, representing a 12% increase from Q2.",
        "faithfulness": 1.0,       # Fully grounded in context
        "answer_relevance": 0.95,  # Directly answers the question
    },
    {
        "id": "TC-002",
        "input": "Who is the CEO?",
        "context": "The company was founded in 2015 by John Smith.",
        "expected_output": "The CEO information is not available in the provided context.",
        "actual_output": "The CEO is John Smith.",  # WRONG: founder ≠ CEO
        "faithfulness": 0.3,       # Inferred incorrectly from context
        "answer_relevance": 0.7,   # Attempted to answer
    },
    {
        "id": "TC-003",
        "input": "What are the main risks for Q4?",
        "context": "Q4 risks include regulatory uncertainty in EU markets and supply chain delays.",
        "expected_output": "Q4 risks include regulatory uncertainty and supply chain delays.",
        "actual_output": "The main Q4 risks are regulatory uncertainty in EU markets, supply chain delays, and potential currency fluctuations.",
        "faithfulness": 0.65,      # 'currency fluctuations' NOT in context = hallucination
        "answer_relevance": 0.9,
    },
    {
        "id": "TC-004",
        "input": "Summarize the financial performance.",
        "context": "Revenue grew 15% YoY. Net income was $8.2M. Operating margin improved to 22%.",
        "expected_output": "Revenue grew 15% YoY with net income of $8.2M and 22% operating margin.",
        "actual_output": "The company showed strong financial performance with 15% revenue growth, $8.2M net income, and an operating margin of 22%.",
        "faithfulness": 1.0,
        "answer_relevance": 0.95,
    },
]

# Evaluate with thresholds
FAITHFULNESS_THRESHOLD = 0.7
RELEVANCE_THRESHOLD = 0.7

print("🧪 DeepEval Test Results:")
print(f"\n{'ID':<10} {'Faithfulness':<15} {'Relevance':<15} {'Status'}")
print("─" * 55)

all_passed = True
for tc in test_cases:
    f_pass = tc["faithfulness"] >= FAITHFULNESS_THRESHOLD
    r_pass = tc["answer_relevance"] >= RELEVANCE_THRESHOLD
    passed = f_pass and r_pass
    if not passed: all_passed = False
    status = "✅ PASS" if passed else "❌ FAIL"
    print(f"{tc['id']:<10} {tc['faithfulness']:<15.2f} {tc['answer_relevance']:<15.2f} {status}")
    if not passed:
        if not f_pass: print(f"           ↳ Faithfulness below {FAITHFULNESS_THRESHOLD}: likely hallucination in output")
        if not r_pass: print(f"           ↳ Relevance below {RELEVANCE_THRESHOLD}: output doesn't answer the question")

print(f"\n{'✅ ALL TESTS PASSED' if all_passed else '❌ SOME TESTS FAILED — deployment blocked'}")

### 🤔 What Happens When Tests Fail?

| Failure | Root Cause | Fix |
|---------|-----------|-----|
| Low faithfulness | Model adds info not in context | Add \"only use provided context\" instruction |
| Low relevance | Model doesn't answer the question | Improve prompt structure |
| Both low | Wrong retrieval + bad generation | Fix retrieval first, then generation |

**Real-world scenario:**  
A customer support chatbot was deployed. It had high relevance (it answered questions) but low faithfulness (it made up product features that didn't exist). Customers got excited about features that didn't exist → support tickets → angry customers.

---

## 2 · RAGAS: Evaluating RAG Pipelines

### 🤔 What is RAGAS?

RAGAS (Retrieval-Augmented Generation Assessment) evaluates the ENTIRE RAG pipeline, not just the LLM:

```
Query → [Retrieval] → [Context] → [Generation] → Answer
          ↑                           ↑
    Context Precision            Faithfulness
    Context Recall               Answer Relevance
```

### RAGAS Metrics Explained

| Metric | Evaluates | Question | Score Meaning |
|--------|----------|---------|---------------|
| **Context Precision** | Retrieval ranking | Are relevant docs at the TOP of results? | 1.0 = perfect ranking |
| **Context Recall** | Retrieval completeness | Did we find ALL relevant docs? | 1.0 = found everything |
| **Faithfulness** | Generation grounding | Is the answer based ONLY on context? | 1.0 = fully grounded |
| **Answer Relevance** | Generation usefulness | Does it answer the actual question? | 1.0 = perfect answer |

### 🤔 Why Evaluate Retrieval and Generation Separately?

Because the fix is different:
- **Bad retrieval + good generation** → Fix your embedding model or chunking strategy
- **Good retrieval + bad generation** → Fix your prompt or switch models
- **Bad both** → Fix retrieval FIRST (generation can't succeed without good context)

In [ ]:
# Cell 3 — Evaluate a mock RAG pipeline with RAGAS-style metrics
#
# We simulate a RAG dataset with:
# - Queries (what the user asked)
# - Retrieved contexts (what the retriever found)
# - Generated answers (what the LLM produced)
# - Ground truth (the correct answer)
#
# Some examples are deliberately BAD to show what failure looks like.

import pandas as pd
import numpy as np

rag_dataset = [
    {
        "query": "What is the maximum context window for GPT-4o?",
        "contexts": ["GPT-4o supports a context window of 128,000 tokens.", "GPT-4 was released by OpenAI."],
        "answer": "GPT-4o supports a maximum context window of 128,000 tokens.",
        "ground_truth": "128,000 tokens",
        "context_precision": 1.0,   # Relevant doc is rank 1
        "context_recall": 1.0,      # Found the doc with the answer
        "faithfulness": 1.0,        # Answer comes from context
        "answer_relevance": 0.95,
    },
    {
        "query": "How does LoRA reduce training costs?",
        "contexts": ["Transformers use attention mechanisms.", "The weather in Tokyo is mild."],
        "answer": "LoRA reduces training costs by only training small adapter matrices instead of the full model.",
        "ground_truth": "LoRA adds small adapter modules that are trained while the base model stays frozen.",
        "context_precision": 0.0,   # ❌ No relevant doc retrieved!
        "context_recall": 0.0,      # ❌ Completely missed
        "faithfulness": 0.0,        # Answer NOT from context (hallucinated, though correct)
        "answer_relevance": 0.85,
    },
    {
        "query": "What databases does Feast support as online stores?",
        "contexts": ["Feast supports Redis, DynamoDB, and SQLite as online stores.", "Feast is a feature store for ML."],
        "answer": "Feast supports Redis, DynamoDB, SQLite, and PostgreSQL as online stores.",
        "ground_truth": "Redis, DynamoDB, and SQLite",
        "context_precision": 1.0,
        "context_recall": 1.0,
        "faithfulness": 0.75,       # ⚠️ PostgreSQL NOT in context = hallucination
        "answer_relevance": 0.9,
    },
    {
        "query": "What is the EU AI Act penalty for non-compliance?",
        "contexts": ["The EU AI Act can impose fines up to 7% of global annual turnover.", "The Act classifies AI systems by risk level."],
        "answer": "The EU AI Act can impose fines of up to 7% of a company's global annual revenue.",
        "ground_truth": "Up to 7% of global annual turnover",
        "context_precision": 1.0,
        "context_recall": 1.0,
        "faithfulness": 1.0,
        "answer_relevance": 0.95,
    },
    {
        "query": "How does KV-cache work in vLLM?",
        "contexts": ["vLLM uses PagedAttention for memory management.", "Docker containers provide isolation."],
        "answer": "vLLM uses PagedAttention to manage the KV-cache efficiently, paging attention blocks similar to OS virtual memory.",
        "ground_truth": "PagedAttention manages KV-cache by paging attention key-value blocks.",
        "context_precision": 0.5,   # Partially relevant doc at rank 1
        "context_recall": 0.5,      # Found PagedAttention but not KV-cache details
        "faithfulness": 0.8,
        "answer_relevance": 0.9,
    },
]

df = pd.DataFrame(rag_dataset)
print("📊 RAG Pipeline Evaluation Results:\n")
print(df[["query", "context_precision", "context_recall", "faithfulness", "answer_relevance"]].to_string(index=False))

print(f"\n📈 Average Scores:")
for metric in ["context_precision", "context_recall", "faithfulness", "answer_relevance"]:
    mean = df[metric].mean()
    status = "✅" if mean >= 0.7 else "❌"
    print(f"   {status} {metric:<22}: {mean:.2f}")

In [ ]:
# Cell 4 — Identify poorly-performing queries
#
# This is the diagnostic step: WHICH queries failed and WHY?
# In production, this drives improvement:
# - Low context_precision → re-chunk documents, improve retrieval
# - Low faithfulness → add guardrails, improve prompt

print("🔍 Queries Needing Attention:\n")

for _, row in df.iterrows():
    issues = []
    if row["context_precision"] < 0.5: issues.append("❌ Bad retrieval ranking")
    if row["context_recall"] < 0.5: issues.append("❌ Missing relevant context")
    if row["faithfulness"] < 0.7: issues.append("⚠️ Hallucination risk")
    if row["answer_relevance"] < 0.7: issues.append("⚠️ Off-topic answer")

    if issues:
        print(f"   Query: \"{row['query']}\"")
        for issue in issues:
            print(f"      {issue}")
        print(f"      Fix: {'Improve retrieval (embeddings/chunking)' if row['context_recall'] < 0.5 else 'Add \"only use provided context\" to prompt'}")
        print()

---

## 3 · Langfuse: Tracing Agentic Workflows

### 🤔 What is Tracing and Why Does It Matter?

A multi-step agent might make 5-10 LLM calls to answer ONE question. When something goes wrong, you need to know EXACTLY which step failed.

```
User query → [Retrieval] → [Reranking] → [Generation] → [Safety check] → Response
                                            ↑
                                    WHERE DID IT FAIL?
```

**Without tracing:** "The chatbot gave a wrong answer. I have no idea why."  
**With tracing:** "Step 3 (Generation) used the wrong context chunk. The retriever returned 5 chunks but the generator only looked at the first 2."

### What Langfuse Captures

| Data | Why | Example |
|------|-----|--------|
| Execution spans | See how long each step takes | Retrieval: 50ms, Generation: 2.1s |
| Input/Output | Debug exact prompts and responses | Prompt: \"...\" → Response: \"...\" |
| Token usage | Track costs per request | 500 input + 200 output = $0.01 |
| Metadata | Link to prompt version, model, etc. | prompt_v3, gpt-4o, temp=0.7 |

### 🤔 How Does the `@observe` Decorator Work?

You add `@observe()` to any function. Langfuse automatically records:
- When the function started/ended
- What arguments it received
- What it returned
- How long it took
- Nested calls (if function A calls function B, both are traced)

In [ ]:
# Cell 5 — Simulate Langfuse tracing for an agentic workflow
#
# In production, you'd use:
#   from langfuse.decorators import observe
#   @observe()
#   def my_function(...):
#
# Here we simulate the tracing output to show what you'd see
# in the Langfuse dashboard.

import time
import uuid
from datetime import datetime

class TracingSimulator:
    """Simulates Langfuse @observe decorator behavior."""

    def __init__(self):
        self.traces = []
        self.current_trace_id = None

    def start_trace(self, name: str):
        self.current_trace_id = str(uuid.uuid4())[:8]
        self.traces.append({"trace_id": self.current_trace_id, "name": name, "spans": [], "start": datetime.utcnow()})

    def log_span(self, name: str, input_data: str, output_data: str, duration_ms: float, tokens_in: int = 0, tokens_out: int = 0, model: str = None):
        span = {
            "span_id": str(uuid.uuid4())[:8], "name": name,
            "input": input_data[:100], "output": output_data[:100],
            "duration_ms": duration_ms, "tokens_in": tokens_in, "tokens_out": tokens_out, "model": model,
        }
        if self.traces:
            self.traces[-1]["spans"].append(span)

    def end_trace(self):
        if self.traces:
            self.traces[-1]["end"] = datetime.utcnow()
            self.traces[-1]["total_ms"] = sum(s["duration_ms"] for s in self.traces[-1]["spans"])
            self.traces[-1]["total_cost"] = sum((s["tokens_in"] * 0.005 + s["tokens_out"] * 0.015) / 1000 for s in self.traces[-1]["spans"])

tracer = TracingSimulator()

# Simulate an agentic workflow
tracer.start_trace("customer_support_agent")
tracer.log_span("query_classification", "What is your return policy?", "category: returns_policy", 45, 20, 5, "gpt-4o-mini")
tracer.log_span("document_retrieval", "returns_policy", "Found 3 relevant docs: policy_v2.md, faq.md, terms.md", 120, 0, 0)
tracer.log_span("context_reranking", "3 docs", "Reranked: policy_v2.md (0.95), faq.md (0.82), terms.md (0.41)", 35, 200, 10, "cohere-rerank")
tracer.log_span("response_generation", "Query + top 2 contexts", "Our return policy allows returns within 30 days of purchase...", 1850, 500, 150, "gpt-4o")
tracer.log_span("safety_check", "Generated response", "SAFE: no PII, no harmful content", 25, 150, 5, "gpt-4o-mini")
tracer.end_trace()

trace = tracer.traces[0]
print(f"🔍 Langfuse Trace: {trace['name']} (ID: {trace['trace_id']})")
print(f"   Total duration: {trace['total_ms']:.0f} ms")
print(f"   Total cost: ${trace['total_cost']:.6f}")
print(f"\n{'Step':<25} {'Duration':<12} {'Tokens':<15} {'Model':<20} {'Output'}")
print("─" * 110)
for span in trace["spans"]:
    tokens = f"{span['tokens_in']}→{span['tokens_out']}" if span['tokens_in'] else "—"
    model = span['model'] or '—'
    print(f"{span['name']:<25} {span['duration_ms']:<12.0f}ms {tokens:<15} {model:<20} {span['output'][:40]}")

In [ ]:
# Cell 6 — Visualize the trace as a dependency graph

trace_visualization = """
🔍 Trace: customer_support_agent
═══════════════════════════════════════════════════════════════════

│ query_classification (45ms) ─── gpt-4o-mini ── 20→5 tokens
│   Input:  "What is your return policy?"
│   Output: "category: returns_policy"
│
├─► document_retrieval (120ms) ── vector_db
│   Input:  "returns_policy"
│   Output: "3 docs found"
│
├─► context_reranking (35ms) ── cohere-rerank ── 200→10 tokens
│   Input:  3 documents
│   Output: policy_v2.md (0.95), faq.md (0.82)
│
├─► response_generation (1850ms) ── gpt-4o ── 500→150 tokens  ← BOTTLENECK
│   Input:  Query + top 2 contexts
│   Output: "Our return policy allows returns within 30 days..."
│
└─► safety_check (25ms) ── gpt-4o-mini ── 150→5 tokens
    Input:  Generated response
    Output: "SAFE"

Total: 2075ms | $0.009 | 5 spans
Bottleneck: response_generation (89% of total time)
"""

print(trace_visualization)

In [ ]:
# Cell 7 — Demonstrate how tracing enables root-cause analysis
#
# Scenario: Production alert — response quality dropped by 20%.
# How do you find the root cause?

print("📋 Root-Cause Analysis with Tracing")
print("="*55)
print("\n🚨 Alert: Answer relevance dropped from 0.9 to 0.72")
print("\nStep 1: Check trace metrics over time")
print("   - Retrieval latency: stable (120ms avg)")
print("   - Context precision: DROPPED from 0.95 to 0.60 ← 🔴")
print("   - Generation faithfulness: stable (0.9)")
print("\nStep 2: Drill into low-scoring traces")
print("   - Retriever returning wrong documents since Feb 28 14:00")
print("   - Embedding model was updated at Feb 28 13:55 ← ROOT CAUSE")
print("\nStep 3: Fix")
print("   - Rollback embedding model to previous version")
print("   - Re-index all documents with the correct model")
print("   - Add embedding model version to trace metadata")
print("\n✅ Without tracing, this would take days. With tracing: 30 minutes.")

In [ ]:
# Cell 8 — Production tracing with @observe (code reference)
#
# This is the ACTUAL code you'd use in production with Langfuse.
# We write it as a reference file.

production_code = '''
from langfuse.decorators import observe, langfuse_context
import litellm

@observe()  # ← This single decorator captures EVERYTHING
def classify_query(query: str) -> str:
    """Classify the user query into a category."""
    response = litellm.completion(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": f"Classify: {query}"}],
    )
    return response.choices[0].message.content

@observe()
def retrieve_context(category: str) -> list:
    """Retrieve relevant documents from vector store."""
    results = chroma_collection.query(query_texts=[category], n_results=3)
    return results["documents"][0]

@observe()
def generate_response(query: str, context: list) -> str:
    """Generate response grounded in retrieved context."""
    context_str = "\\n".join(context)
    response = litellm.completion(
        model="gpt-4o",
        messages=[{"role": "system", "content": f"Answer using ONLY this context:\\n{context_str}"},
                  {"role": "user", "content": query}],
    )
    return response.choices[0].message.content

@observe()  # ← Top-level function automatically nests all sub-calls
def agent_pipeline(query: str) -> str:
    """Full agentic pipeline — all sub-calls are automatically traced."""
    category = classify_query(query)
    context = retrieve_context(category)
    response = generate_response(query, context)
    langfuse_context.update_current_observation(metadata={"prompt_version": "v3"})
    return response
'''

print("📄 Production Langfuse Integration:")
print(production_code)
print("\n💡 Key: @observe() on the top-level function automatically")
print("   creates a trace tree with nested spans for all sub-calls.")

---

## 🎯 Summary & Key Takeaways

| Step | Action | Tool | Why |
|------|--------|------|-----|
| 1 | LLM output unit tests (faithfulness, relevance) | DeepEval | Block bad models in CI/CD |
| 2 | RAG pipeline evaluation (precision, recall) | RAGAS | Diagnose retrieval vs generation issues |
| 3 | Multi-step execution tracing | Langfuse | Root-cause analysis in production |

### 🧠 Key Questions

1. **"How many test cases do I need?"** → Start with 50-100 covering your most critical use cases. Expand based on production failures.
2. **"Should I test every response in production?"** → No. Sample 5-10% for evaluation. 100% tracing is fine (it's lightweight).
3. **"Which metric matters most?"** → Faithfulness for factual systems. Relevance for Q&A. Context recall for RAG.
4. **"Is LLM-as-a-judge reliable?"** → ~85% agreement with humans. Always validate with human review for high-stakes decisions.

**← Back to** `07_governance_security.ipynb` — The complete curriculum is now covered.